# TFM: Análisis de Políticas de Sostenibilidad mediante técnicas de Argumentacion Computacional

## Clasificación de relaciones con MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli from microsoft/DeBERTa-v3-base

Model page: https://huggingface.co/MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli

Entailment is calculated in both directions (forward probs (Arg1->Arg2) and backward probs (Arg2->Arg1)), then the label is decided with the following criteria:
   
Bidirectional decision:
  - Rephrase: high entailment both ways, low contradiction
  - Attack: contradiction high either way
  - Support: entailment high at least one way, and not clearly neutral/contradictory
  - No Relationship: otherwise

#Keywords

In [1]:
import os
import re
import math
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import gc

process_rel_path = r"/kaggle/working/TFM/Data/Relationships Keywords"

model_name = "MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli"
device = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 50                 
MODEL_TAG = "deberta"      
VALID_OUT = {"Support", "Attack", "Rephrase", "No Relationship"}

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)
model.eval()

# figure out label 
id2label = {int(k): v.lower() for k, v in model.config.id2label.items()}
label2id = {v: int(k) for k, v in id2label.items()}

print(f'DeBERTa v3 base mnli labels {label2id}')

IDX_ENT = label2id["entailment"]
IDX_CON = label2id["contradiction"]
IDX_NEU = label2id["neutral"]

# Heuristic thresholds 
ENT_THR       = 0.50   
CONTR_THR     = 0.50   
REPHRASE_THR  = 0.75  
MAX_NEUTRAL   = 0.70   

# Margins: “X wins by at least this much”
SUPPORT_MARGIN = 0.10  # p_ent - p_con must exceed this (either direction)
ATTACK_MARGIN  = 0.10  # p_con - p_ent must exceed this (either direction)

tokenizer_config.json: 0.00B [00:00, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/286 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

2025-08-25 11:32:56.085797: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1756121576.262452      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1756121576.312794      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


model.safetensors:   0%|          | 0.00/369M [00:00<?, ?B/s]

DeBERTa v3 base mnli labels {'entailment': 0, 'neutral': 1, 'contradiction': 2}


In [2]:
!git clone https://github.com/camipalo/TFM.git

Cloning into 'TFM'...
remote: Enumerating objects: 2459, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (10/10), done.
remote: Total 2459 (delta 9), reused 6 (delta 5), pack-reused 2444 (from 2)
Receiving objects: 100% (2459/2459), 97.50 MiB | 19.08 MiB/s, done.
Resolving deltas: 100% (2065/2065), done.
Updating files: 100% (1061/1061), done.


In [3]:
def preprocess_text(s):
    if not isinstance(s, str):
        return ""
    s = s.strip().lower()
    s = re.sub(r"\s+", " ", s)
    return s

@torch.no_grad()
def nli_probs(pairs, max_length: int):
    if not pairs:
        return np.zeros((0,3)), np.zeros((0,3))

    a1 = [preprocess_text(x[0]) for x in pairs]
    a2 = [preprocess_text(x[1]) for x in pairs]

    # forward: premise=a1, hypothesis=a2
    enc_f = tokenizer(a1, a2, return_tensors="pt", padding=True, truncation=True,
                      max_length=max_length).to(device)
    logits_f = model(**enc_f).logits
    prob_f = torch.softmax(logits_f, dim=-1).detach().cpu().numpy()

    # backward: premise=a2, hypothesis=a1
    enc_b = tokenizer(a2, a1, return_tensors="pt", padding=True, truncation=True,
                      max_length=max_length).to(device)
    logits_b = model(**enc_b).logits
    prob_b = torch.softmax(logits_b, dim=-1).detach().cpu().numpy()

    return prob_f, prob_b

def decide_label(p_ent_f, p_con_f, p_neu_f, p_ent_b, p_con_b, p_neu_b):
    # 1) Hard guard: if both sides look very neutral, say NR
    if max(p_neu_f, p_neu_b) >= MAX_NEUTRAL:
        return "No Relationship"

    # 2) Rephrase: bi-directional entailment and low contradiction
    if min(p_ent_f, p_ent_b) >= REPHRASE_THR and max(p_con_f, p_con_b) <= (1 - REPHRASE_THR):
        return "Rephrase"

    # 3) Attack: contradiction wins with margin OR strong contradiction
    if (
        (p_con_f - p_ent_f) >= ATTACK_MARGIN or
        (p_con_b - p_ent_b) >= ATTACK_MARGIN or
        p_con_f >= CONTR_THR or
        p_con_b >= CONTR_THR
    ):
        return "Attack"

    # 4) Support: entailment wins with margin OR clears lowered threshold
    if (
        (p_ent_f - p_con_f) >= SUPPORT_MARGIN or
        (p_ent_b - p_con_b) >= SUPPORT_MARGIN or
        p_ent_f >= ENT_THR or
        p_ent_b >= ENT_THR
    ):
        return "Support"

    # 5) Fallback
    return "No Relationship"


def free_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def compute_max_length(input_dir, prefix_substring, safety_limit=512):
    max_len = 0
    files = [f for f in os.listdir(input_dir) if f.endswith(".csv") and prefix_substring in f]
    for fn in files:
        path = os.path.join(input_dir, fn)
        try:
            df = pd.read_csv(path)
        except Exception:
            continue
        if "SDGarg1" not in df.columns or "SDGarg2" not in df.columns:
            continue
        for a, b in zip(df["SDGarg1"], df["SDGarg2"]):
            a = preprocess_text(str(a) if pd.notna(a) else "")
            b = preprocess_text(str(b) if pd.notna(b) else "")
            ids = tokenizer(a, b, truncation=False, padding=False)["input_ids"]
            max_len = max(max_len, len(ids))
    if max_len == 0:
        max_len = 128
    return min(max_len, safety_limit)

def classify_relationships_deberta(input_dir, prefix_substring, model_tag=MODEL_TAG):
    files = [f for f in os.listdir(input_dir) if f.endswith(".csv") and prefix_substring in f]
    if not files:
        print(f"No CSVs found in '{input_dir}' containing '{prefix_substring}'.")
        return

    MAX_LENGTH = compute_max_length(input_dir, prefix_substring, safety_limit=512)
    print(f"Using MAX_LENGTH = {MAX_LENGTH}")

    rel_col = f"rel_{model_tag}"

    for fn in files:
        path = os.path.join(input_dir, fn)
        print(f"\nProcessing: {path}")
        df = pd.read_csv(path)

        if "SDGarg1" not in df.columns or "SDGarg2" not in df.columns:
            print(f"  Skipped (missing SDGarg1/SDGarg2): {fn}")
            continue

        df[rel_col] = ""

        n = len(df)
        total_done = 0
        first_examples = []

        batches = math.ceil(n / BATCH_SIZE)
        for bi in range(batches):
            s = bi * BATCH_SIZE
            e = min((bi + 1) * BATCH_SIZE, n)
            chunk = df.iloc[s:e]

            pairs = list(zip(chunk["SDGarg1"].astype(str).tolist(),
                             chunk["SDGarg2"].astype(str).tolist()))
            try:
                prob_f, prob_b = nli_probs(pairs, max_length=MAX_LENGTH)
                labels = []
                for i in range(len(pairs)):
                    pf = prob_f[i]; pb = prob_b[i]
                    p_ent_f, p_neu_f, p_con_f = pf[IDX_ENT], pf[IDX_NEU], pf[IDX_CON]
                    p_ent_b, p_neu_b, p_con_b = pb[IDX_ENT], pb[IDX_NEU], pb[IDX_CON]
                    lab = decide_label(p_ent_f, p_con_f, p_neu_f, p_ent_b, p_con_b, p_neu_b)
                    labels.append(lab)
            except Exception as ex:
                print(f"  Batch {bi+1}/{batches} error: {ex}. Marking 'No Relationship'.")
                labels = ["No Relationship"] * len(pairs)

            df.loc[chunk.index, rel_col] = labels

            # first 5 examples
            for (a1, a2), lab in zip(pairs, labels):
                if len(first_examples) < 5:
                    first_examples.append((a1, a2, lab))
                elif len(first_examples) == 5:
                    print("Sample predictions (first 5):\n")
                    for _a1, _a2, _lab in first_examples:
                        print(f"- Arg1: {_a1}\n- Arg2: {_a2}\n  Label: {_lab}\n")
                    first_examples.append((a1, a2, lab))

            total_done += len(pairs)
            if total_done % 100 < BATCH_SIZE:
                print(f"  Progress: {total_done}/{n} relations classified...")

        # Final label validation 
        mask_nan = df["SDGarg1"].isna() | df["SDGarg2"].isna()
        df.loc[mask_nan, rel_col] = "No Relationship"
        bad = ~df[rel_col].isin(VALID_OUT)
        if bad.any():
            df.loc[bad, rel_col] = "No Relationship"

        df.to_csv(path, index=False, encoding="utf-8")
        free_cuda()
        print(f"Saved: {path}")
    return df


## GLOBAL SDG 2023 

#### Qwen2.5 3B extraction

In [4]:
prefix = "intra_goalGLOBAL_SGD2023_qwen2.5-3b"   

args_classified = classify_relationships_deberta(process_rel_path, prefix)
display(args_classified.sample(5))

args_classified["rel_deberta"].value_counts()

Using MAX_LENGTH = 140

Processing: /kaggle/working/TFM/Data/Relationships Keywords/intra_goalGLOBAL_SGD2023_qwen2.5-3b.csv
Sample predictions (first 5):

- Arg1: At their core, the SDGs are an investment agenda: it is critical that UN Member States adopt and implement the SDG Stimulus and support a comprehensive reform of the global financial architecture.
- Arg2: Investing in statistical capacity, science, and data literacy are important priorities for achieving the SDGs.
  Label: No Relationship

- Arg1: At their core, the SDGs are an investment agenda: it is critical that UN Member States adopt and implement the SDG Stimulus and support a comprehensive reform of the global financial architecture.
- Arg2: At the global level, averaging across countries, not a single SDG is currently projected to be met by 2030, with the poorest countries struggling the most.
  Label: No Relationship

- Arg1: At their core, the SDGs are an investment agenda: it is critical that UN Member States adopt

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,SDGarg1,SDGarg2,id_arg1,id_arg2,rel,rel_flanT5,rel_robertaL,rel_deberta
123,Investing in the SDGs is an investment agenda....,All countries should use the half-way momentum...,0_5,0_9,NaN,Support,Support,No Relationship
382,"All countries, poorer and richer alike, should...",industrial,8_0,8_4,NaN,No Relationship,No Relationship,No Relationship
42,"Investing in statistical capacity, science, an...",The SDGs require long-term directed change and...,0_1,0_18,NaN,Support,Support,No Relationship
414,"China has reiterated its support for the SDGs,...","Provincial, metropolitan, and city governments...",11_2,11_3,NaN,Support,Support,No Relationship
168,"The United States, as the world’s biggest econ...","We conclude by underscoring the vital, life-af...",0_7,0_15,NaN,Support,Support,No Relationship


rel_deberta
No Relationship    505
Attack               1
Rephrase             1
Name: count, dtype: int64

In [5]:
prefix = "cross_goalGLOBAL_SGD2023_qwen2.5-3b"   

args_classified = classify_relationships_deberta(process_rel_path, prefix)
display(args_classified.sample(5))

args_classified["rel_deberta"].value_counts()

Using MAX_LENGTH = 149

Processing: /kaggle/working/TFM/Data/Relationships Keywords/cross_goalGLOBAL_SGD2023_qwen2.5-3b.csv
Sample predictions (first 5):

- Arg1: At their core, the SDGs are an investment agenda: it is critical that UN Member States adopt and implement the SDG Stimulus and support a comprehensive reform of the global financial architecture.
- Arg2: Increased funding from the multilateral development banks (MDBs) and public development banks (PDBs) to low- and middle-income countries, linked to investments in the SDGs;
  Label: No Relationship

- Arg1: At their core, the SDGs are an investment agenda: it is critical that UN Member States adopt and implement the SDG Stimulus and support a comprehensive reform of the global financial architecture.
- Arg2: Many of them lack an adequately high SDG commitment, and almost all lack access to the necessary financial means to implement the SDGs.
  Label: No Relationship

- Arg1: At their core, the SDGs are an investment agenda: 

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,SDGarg1,SDGarg2,id_arg1,id_arg2,rel,rel_flanT5,rel_robertaL,rel_deberta
92,Many of them lack an adequately high SDG commi...,Many rich countries also face a significant ch...,0_11,2_0,NaN,Support,Support,No Relationship
3055,"The United States, as the world’s biggest econ...",Sustainable cities: urban infrastructure and s...,8_1,11_0,NaN,Support,Support,No Relationship
2490,We see across societies that inequalities are ...,Reform current institutional frameworks and de...,4_1,17_4,NaN,Support,No Relationship,No Relationship
394,"By contrast, Lebanon, Yemen, Papua New Guinea,...",Although on average the world has made some pr...,0_17,6_2,NaN,No Relationship,Support,No Relationship
3271,"The European Green Deal (EGD), which is exempl...",The SDSN is highly committed to supporting glo...,9_1,15_2,NaN,Support,Support,No Relationship


rel_deberta
No Relationship    3909
Attack               24
Rephrase             21
Support               4
Name: count, dtype: int64

#### Gemma3 27B extraction

In [6]:
prefix = "intra_goalGLOBAL_SGD2023_gemma3-27b"   
rel_col = f"rel_{model_name}"

args_classified = classify_relationships_deberta(process_rel_path, prefix)
display(args_classified.sample(5))

args_classified["rel_deberta"].value_counts()

Using MAX_LENGTH = 168

Processing: /kaggle/working/TFM/Data/Relationships Keywords/intra_goalGLOBAL_SGD2023_gemma3-27b.csv
Sample predictions (first 5):

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
- Arg2: Despite this alarming development, the SDGs are still achievable.
  Label: No Relationship

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
- Arg2: At their core, the SDGs are an investment agenda: it is critical that UN Member States adopt and implement the SDG Stimulus and support a comprehensive reform of the global financial architecture.
  Label: No Relationship

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
- Arg2: To achieve the SDGs the world must both alter its current investment patterns and increase the overall volume of investments.
  Label: No Relationship

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
- Arg2: The St

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,SDGarg1,SDGarg2,id_arg1,id_arg2,rel,rel_flanT5,rel_robertaL,rel_deberta
2864,"All countries, poorer and richer alike, should...",national governments must also work with subna...,17_3,17_10,NaN,Support,Support,No Relationship
648,"Unless the SDGs are actively pursued, geophysi...","At their core, the SDGs are an investment agenda.",0_15,0_19,NaN,No Relationship,Support,No Relationship
1484,"And yet, the truth remains that hundreds of mi...",Governments are only now mapping out pathways ...,4_2,4_9,NaN,No Relationship,No Relationship,No Relationship
559,The SDGs are seriously off track (Figure 1.1).,"Second, UN Member States, starting with the G2...",0_12,0_38,NaN,Support,Support,No Relationship
2011,"4.\t Sustainable ecosystems, sustainable agric...",Others include slowing or stopping the global ...,13_1,13_9,NaN,No Relationship,Support,No Relationship


rel_deberta
No Relationship    3285
Attack               63
Support               3
Rephrase              3
Name: count, dtype: int64

In [7]:
prefix = "cross_goalGLOBAL_SGD2023_gemma3-27b"   

args_classified = classify_relationships_deberta(process_rel_path, prefix)
display(args_classified.sample(5))

args_classified["rel_deberta"].value_counts()

Using MAX_LENGTH = 172

Processing: /kaggle/working/TFM/Data/Relationships Keywords/cross_goalGLOBAL_SGD2023_gemma3-27b.csv
Sample predictions (first 5):

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
- Arg2: Greatly increased funding for national and subnational governments and private businesses in the emerging economies, especially the low-income countries (LICs) and lower-middle-income countries (LMICs), to carry out needed SDG actions;
  Label: Attack

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
- Arg2: to give all people the skills and knowledge to end poverty, protect the environment, and build peaceful and inclusive societies.
  Label: No Relationship

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
- Arg2: Although all governments are in principle committed to economic justice as enshrined in the Universal Declaration of Human Rights, and to the SDG tenets of ‘l

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,SDGarg1,SDGarg2,id_arg1,id_arg2,rel,rel_flanT5,rel_robertaL,rel_deberta
36466,SDG 16 recognizes the vital role of peaceful a...,One of the consistent findings of the SDSN is ...,16_9,17_19,NaN,Support,Support,No Relationship
6046,The Stimulus’ urgent objective is to address t...,the “Code of Conduct on Responsible Food Busin...,0_4,12_0,NaN,Support,No Relationship,No Relationship
30842,the promotion of key enablers such as digital ...,SDG 17 (Partnerships for the Goals) calls on a...,10_8,17_28,NaN,Support,Rephrase,No Relationship
28824,2. Infrastructure: Energy production and distr...,"All countries, poorer and richer alike, should...",9_1,17_3,NaN,Support,Support,No Relationship
33833,"Global spending on armaments, estimated at US$...",Unifying international business ecosystems cou...,13_2,17_16,NaN,No Relationship,No Relationship,No Relationship


rel_deberta
No Relationship    36485
Attack               265
Rephrase              66
Support               16
Name: count, dtype: int64

#### Gemma3 4B extraction

In [8]:
prefix = "intra_goalGLOBAL_SGD2023_gemma3-4b"   

args_classified = classify_relationships_deberta(process_rel_path, prefix)
display(args_classified.sample(5))

args_classified["rel_deberta"].value_counts()

Using MAX_LENGTH = 248

Processing: /kaggle/working/TFM/Data/Relationships Keywords/intra_goalGLOBAL_SGD2023_gemma3-4b.csv
Sample predictions (first 5):

- Arg1: the SDGs are seriously off track
- Arg2: the SDGs are still achievable
  Label: Attack

- Arg1: the SDGs are seriously off track
- Arg2: it is critical that UN Member States adopt and implement the SDG Stimulus
  Label: Attack

- Arg1: the SDGs are seriously off track
- Arg2: To achieve the SDGs the world must both alter its current investment patterns and increase the overall volume of investments
  Label: No Relationship

- Arg1: the SDGs are seriously off track
- Arg2: Revise the credit rating system and debt sustainability metrics to facilitate long-term sustainable development
  Label: No Relationship

- Arg1: the SDGs are seriously off track
- Arg2: Align private business investment flows with the SDGs, through improved national planning, regulation, reporting, and oversight
  Label: Attack

  Progress: 100/6976 relation

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,SDGarg1,SDGarg2,id_arg1,id_arg2,rel,rel_flanT5,rel_robertaL,rel_deberta
5216,"UN Member States should adopt an SDG Stimulus,...","Deep, chronic, and crippling under-investment ...",10_3,10_9,NaN,Support,Rephrase,No Relationship
3677,"investing in statistical capacity, science, an...",The SDSN Framework of nearly 90% of global fis...,1_3,1_9,NaN,No Relationship,No Relationship,No Relationship
4804,Local governments have the front-line responsi...,SDSN is working closely with the UNESCO SDG 4 ...,4_7,4_9,NaN,Support,Support,No Relationship
3046,all UN Member States will have presented a VNR...,Each of these challenges requires large-scale ...,0_52,0_57,NaN,No Relationship,No Relationship,No Relationship
408,Revise the credit rating system and debt susta...,It is difficult to assess whether the adoption...,0_4,0_83,NaN,No Relationship,No Relationship,No Relationship


rel_deberta
No Relationship    6821
Attack              149
Rephrase              3
Support               3
Name: count, dtype: int64

In [9]:
prefix = "cross_goalGLOBAL_SGD2023_gemma3-4b"   

args_classified = classify_relationships_deberta(process_rel_path, prefix)
display(args_classified.sample(5))

args_classified["rel_deberta"].value_counts()

Using MAX_LENGTH = 267

Processing: /kaggle/working/TFM/Data/Relationships Keywords/cross_goalGLOBAL_SGD2023_gemma3-4b.csv
Sample predictions (first 5):

- Arg1: the SDGs are seriously off track
- Arg2: To achieve the SDGs the world must both alter its current investment patterns and increase the overall volume of investments.
  Label: Attack

- Arg1: the SDGs are seriously off track
- Arg2: Greatly increase funding to national and subnational governments and private businesses, especially in LICs and LMICs, to carry out needed SDG investments.
  Label: No Relationship

- Arg1: the SDGs are seriously off track
- Arg2: Revise liquidity structures for LICs and LMICs, especially regarding sovereign debts, to forestall self-fulfilling banking and balance-of-payments crises;
  Label: No Relationship

- Arg1: the SDGs are seriously off track
- Arg2: investing in statistical capacity, science, and data literacy are important priorities for achieving the SDGs
  Label: No Relationship

- Arg1: 

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,SDGarg1,SDGarg2,id_arg1,id_arg2,rel,rel_flanT5,rel_robertaL,rel_deberta
27204,trade more sustainable and more consistent wit...,Some countries specifically refer to the SDGs ...,1_25,2_19,NaN,Support,Support,No Relationship
55058,globalized trade rules for ‘cleantech’ could a...,Sustainable capital assets are long-lasting ca...,8_6,9_6,NaN,Support,Support,No Relationship
46396,achieving universal health coverage and ensuri...,This scorecard shows that many countries aroun...,3_11,16_30,NaN,Support,Support,No Relationship
55896,Increased funding from the multilateral develo...,"The EGD embraces an EU-wide set of goals, time...",8_2,13_10,NaN,Support,Rephrase,No Relationship
43762,Governments are only now mapping out pathways ...,actions by governments at all levels to ensure...,3_5,4_2,NaN,Support,Support,No Relationship


rel_deberta
No Relationship    70061
Attack              1025
Rephrase             129
Support               19
Name: count, dtype: int64

#### Llama3.3 70B extraction

In [10]:
prefix = "intra_goalGLOBAL_SGD2023_llama3.3-70b"   

args_classified = classify_relationships_deberta(process_rel_path, prefix)
display(args_classified.sample(5))

args_classified["rel_deberta"].value_counts()

Using MAX_LENGTH = 216

Processing: /kaggle/working/TFM/Data/Relationships Keywords/intra_goalGLOBAL_SGD2023_llama3.3-70b.csv
Sample predictions (first 5):

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
- Arg2: Since the outbreak of the pandemic in 2020 and other simultaneous crises, SDG progress has stalled globally.
  Label: No Relationship

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
- Arg2: The world is off track, but that is all the more reason to double down on the SDGs.
  Label: No Relationship

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
- Arg2: At their core, the SDGs are an investment agenda: it is critical that UN Member States adopt and implement the SDG Stimulus and support a comprehensive reform of the global financial architecture.
  Label: No Relationship

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
- Arg2: To 

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,SDGarg1,SDGarg2,id_arg1,id_arg2,rel,rel_flanT5,rel_robertaL,rel_deberta
2529,"We conclude by underscoring the vital, life-af...",As detailed in the Europe Sustainable Developm...,0_52,0_60,NaN,No Relationship,No Relationship,No Relationship
3006,SDSN’s mission was fourfold: (i) scholarly res...,SDG target 4.1 calls for universal access to 1...,4_2,4_16,NaN,Support,No Relationship,No Relationship
492,"All countries, poorer and richer alike, should...",International financing flows should be aligne...,0_6,0_70,NaN,Support,Support,No Relationship
2219,Long-term investment plans are essential for n...,"Provincial, metropolitan, and city governments...",0_41,0_47,NaN,Support,Support,No Relationship
198,"The world is off track, but that is all the mo...","Second, UN Member States, starting with the G2...",0_2,0_54,NaN,Support,Support,No Relationship


rel_deberta
No Relationship    5885
Attack               54
Support              10
Rephrase              2
Name: count, dtype: int64

In [11]:
prefix = "cross_goalGLOBAL_SGD2023_llama3.3-70b"   

args_classified = classify_relationships_deberta(process_rel_path, prefix)
display(args_classified.sample(5))

args_classified["rel_deberta"].value_counts()

Using MAX_LENGTH = 281

Processing: /kaggle/working/TFM/Data/Relationships Keywords/cross_goalGLOBAL_SGD2023_llama3.3-70b.csv
Sample predictions (first 5):

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
- Arg2: The grim reality is that at the midpoint of the 2030 Agenda, the SDGs are far off track.
  Label: Support

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
- Arg2: Similarly, extreme poverty can lead to a collapse of tax revenues, followed by government bankruptcy and further economic collapse, a syndrome that now threatens dozens of poor countries.
  Label: No Relationship

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
- Arg2: Most of the low-income and lower-middle income countries, home to more than the half of humanity, face major challenges in achieving most of the SDGs by 2030.
  Label: No Relationship

- Arg1: At the midpoint of the 2030 Agenda, all of the SDG

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,SDGarg1,SDGarg2,id_arg1,id_arg2,rel,rel_flanT5,rel_robertaL,rel_deberta
20179,One of the consistent findings of the SDSN is ...,"Thus, multilateralism and investments in globa...",0_46,17_39,NaN,Support,Support,No Relationship
10052,"According to major international studies, few ...",This metric is particularly useful for assessi...,0_11,11_9,NaN,No Relationship,No Relationship,No Relationship
14156,The SDGs are woefully underfunded at home and ...,Inland fisheries are also experiencing similar...,0_31,14_2,NaN,No Relationship,No Relationship,No Relationship
48553,HICs are able to mobilize vast financial resou...,Regional cooperation and sustainable developme...,10_9,17_20,NaN,Support,Support,No Relationship
11458,Further investment is needed in statistical ca...,"Moreover, regional cooperation is needed to pr...",0_9,13_25,NaN,Support,Support,No Relationship


rel_deberta
No Relationship    57952
Attack               251
Rephrase              94
Support               13
Name: count, dtype: int64

#### Deepseek r1 70B extraction

In [12]:
prefix = "intra_goalGLOBAL_SGD2023_deepseek-r1-70b"   

args_classified = classify_relationships_deberta(process_rel_path, prefix)
display(args_classified.sample(5))

args_classified["rel_deberta"].value_counts()

Using MAX_LENGTH = 235

Processing: /kaggle/working/TFM/Data/Relationships Keywords/intra_goalGLOBAL_SGD2023_deepseek-r1-70b.csv
Sample predictions (first 5):

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
- Arg2: Despite this alarming development, the SDGs are still achievable.
  Label: No Relationship

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
- Arg2: The world is off track, but that is all the more reason to double down on the SDGs.
  Label: No Relationship

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
- Arg2: To achieve the SDGs the world must both alter its current investment patterns and increase the overall volume of investments.
  Label: No Relationship

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
- Arg2: All countries, poorer and richer alike, should use the half-way momentum to self-critically review and revise thei

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,SDGarg1,SDGarg2,id_arg1,id_arg2,rel,rel_flanT5,rel_robertaL,rel_deberta
4409,"societal polarization, populism, and growing g...","To reduce inequalities, governments also need ...",10_2,10_20,NaN,Support,Support,No Relationship
1880,Dire shortfalls in meeting the SDGs,The SDGs are not only a public policy framewor...,0_26,0_74,NaN,Support,Support,No Relationship
1956,None of their objectives are beyond our reach....,Most of the low-income and lower-middle income...,0_28,0_39,NaN,Support,Support,No Relationship
2909,Our special emphasis is on pathway analysis to...,The SDGs require long-term directed change and...,0_49,0_68,NaN,No Relationship,Support,No Relationship
1242,Current geopolitical tensions are hindering SD...,The urgent objective of the SDG Stimulus is to...,0_16,0_51,NaN,Support,Support,No Relationship


rel_deberta
No Relationship    6408
Attack              112
Support              15
Rephrase              2
Name: count, dtype: int64

In [13]:
prefix = "cross_goalGLOBAL_SGD2023_deepseek-r1-70b"   

args_classified = classify_relationships_deberta(process_rel_path, prefix)
display(args_classified.sample(5))

args_classified["rel_deberta"].value_counts()

Using MAX_LENGTH = 255

Processing: /kaggle/working/TFM/Data/Relationships Keywords/cross_goalGLOBAL_SGD2023_deepseek-r1-70b.csv
Sample predictions (first 5):

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
- Arg2: The grim reality is that at the midpoint of the 2030 Agenda, the SDGs are far off track.
  Label: Support

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
- Arg2: At the global level, averaging across countries, not a single SDG is currently projected to be met by 2030, with the poorest countries struggling the most.
  Label: No Relationship

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
- Arg2: 1. Increased funding from the multilateral develop-ment banks (MDBs) and public development banks (PDBs) to low- and middle-income countries, linked to investments in the SDGs;
  Label: Attack

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off 

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,SDGarg1,SDGarg2,id_arg1,id_arg2,rel,rel_flanT5,rel_robertaL,rel_deberta
47162,Governments generally finance basic scientific...,UNEP estimates that 84 percent of Parties to t...,9_3,13_22,NaN,Support,No Relationship,No Relationship
12744,Greatly increased funding for national and sub...,"The European Green Deal (EGD), which is exempl...",0_11,12_1,NaN,Support,Support,No Relationship
47169,The Horizon Europe program and EU Missions in ...,Even taking the net-zero pledges of many count...,9_4,13_5,NaN,No Relationship,No Relationship,No Relationship
50532,LICs and LMICs score more highly on political ...,"According to IMF estimates in 2019, the financ...",10_22,13_16,NaN,Support,Support,No Relationship
25407,Peace and global cooperation mean nothing less...,Each requires a financing strategy to underpin...,1_12,8_9,NaN,No Relationship,No Relationship,No Relationship


rel_deberta
No Relationship    60434
Attack               381
Rephrase             121
Support               55
Name: count, dtype: int64